# Short-horizon rebuild notebook (1h, 6h, 12h, 24h)

This notebook rebuilds the eight-model comparison from the original project using the same overall methods, but with shorter forecast horizons.

The model set is:

1. Elastic Net  
2. Random Forest  
3. LSTM  
4. CNN–GRU–LSTM  
5. GraphWaveNet  
6. GraphWaveNet–GRU  
7. GraphWaveNet–LSTM  
8. GraphWaveNet–GRU–LSTM  

The methodological goal is to keep the project aligned with the original notebooks:

- same split dates
- same station set
- same feature channels
- same train-only scaling
- same strict sliding-window logic
- same evaluation style in original traffic-flow units

The only forecasting change is:

- old setup: `OUT_LEN = 72`, report `12, 24, 48, 72`
- new setup: `OUT_LEN = 24`, report `1, 6, 12, 24`


In [1]:
from pathlib import Path
import copy
import gc
import json
import random
import time

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import MultiTaskElasticNet
from sklearn.ensemble import RandomForestRegressor
from joblib import Parallel, delayed

from IPython.display import display


def set_seed(seed: int = 42, deterministic: bool = True):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


SEED = 42
set_seed(SEED, deterministic=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# -------------------------
# Raw files
# -------------------------
TRAFFIC_CSV = Path("cleaned_traffic_data.csv")
META_XLSX   = Path("pems_output.xlsx")

# -------------------------
# Existing strict artifact from the original project
# -------------------------
BASE_STRICT_DATASET = Path("artifacts/pems_graph_dataset_strict.npz")

# -------------------------
# Split boundaries (same as original project)
# -------------------------
TRAIN_END = pd.Timestamp("2024-11-15 23:59:59")
VAL_END   = pd.Timestamp("2024-11-30 23:59:59")

# -------------------------
# Short-horizon setup
# -------------------------
IN_LEN = 24
OUT_LEN = 24
EVAL_HORIZONS = [1, 6, 12, 24]

assert max(EVAL_HORIZONS) <= OUT_LEN, "The largest evaluation horizon must be <= OUT_LEN."

SHORT_TAG = "short_h1_6_12_24"

OUT_DIR = Path("artifacts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SHORT_DATASET = OUT_DIR / f"pems_graph_dataset_strict_{SHORT_TAG}.npz"
RESULTS_SUMMARY_SHORT = OUT_DIR / f"results_summary_{SHORT_TAG}.csv"

print("BASE_STRICT_DATASET:", BASE_STRICT_DATASET)
print("SHORT_DATASET:", SHORT_DATASET)
print("RESULTS_SUMMARY_SHORT:", RESULTS_SUMMARY_SHORT)

Torch: 2.1.1+cu121
Device: cpu
BASE_STRICT_DATASET: artifacts/pems_graph_dataset_strict.npz
SHORT_DATASET: artifacts/pems_graph_dataset_strict_short_h1_6_12_24.npz
RESULTS_SUMMARY_SHORT: artifacts/results_summary_short_h1_6_12_24.csv


## Rebuild the strict dataset for 24-step forecasting

This is the key short-horizon conversion step.

To stay methodologically faithful to the original project, we do not rebuild the whole cleaned dataset from scratch here.  
Instead, we:

- load the already-built strict artifact from the original notebook
- keep the same `X`, `Y`, adjacency, stations, timestamps, and train-only statistics
- recompute only the valid start indices and split assignments for `OUT_LEN = 24`

That way, the study changes only in the forecast length and evaluation horizons.

In [2]:
assert BASE_STRICT_DATASET.exists(), (
    f"Missing {BASE_STRICT_DATASET}. "
    "Run the original notebook once so the strict base artifact exists."
)

base = np.load(BASE_STRICT_DATASET, allow_pickle=True)

X_base = base["X"].astype(np.float32)
Y_base = base["Y"].astype(np.float32)
A_base = base["A"].astype(np.float32)
stations_base = base["stations"]
timestamps_base = pd.to_datetime(base["timestamps"])

flow_mean_base = base["flow_mean"].astype(np.float32)
flow_std_base  = base["flow_std"].astype(np.float32)
speed_mean_base = base["speed_mean"].astype(np.float32)
speed_std_base  = base["speed_std"].astype(np.float32)

base_in_len = int(np.array(base["in_len"]).item())
print("Base strict artifact loaded.")
print("Base X:", X_base.shape, "Base Y:", Y_base.shape)
print("Base IN_LEN:", base_in_len)

assert base_in_len == IN_LEN, (
    f"Expected base IN_LEN={IN_LEN}, found {base_in_len}. "
    "This notebook assumes the original project used 24 hours of input."
)

T_total = X_base.shape[0]
max_start = T_total - (IN_LEN + OUT_LEN) + 1
starts = np.arange(max_start, dtype=np.int32)

out_start_times = timestamps_base[starts + IN_LEN]
out_end_times   = timestamps_base[starts + IN_LEN + OUT_LEN - 1]

train_starts_short = starts[out_end_times <= TRAIN_END]
val_starts_short   = starts[(out_start_times > TRAIN_END) & (out_end_times <= VAL_END)]
test_starts_short  = starts[out_start_times > VAL_END]

print("Short-horizon strict window counts:")
print("train:", len(train_starts_short))
print("val:  ", len(val_starts_short))
print("test: ", len(test_starts_short))

np.savez_compressed(
    SHORT_DATASET,
    X=X_base.astype(np.float32),
    Y=Y_base.astype(np.float32),
    A=A_base.astype(np.float32),
    stations=stations_base,
    timestamps=np.array(timestamps_base.astype("datetime64[ns]")),
    train_starts=train_starts_short,
    val_starts=val_starts_short,
    test_starts=test_starts_short,
    in_len=np.array([IN_LEN], dtype=np.int32),
    out_len=np.array([OUT_LEN], dtype=np.int32),
    flow_mean=flow_mean_base,
    flow_std=flow_std_base,
    speed_mean=speed_mean_base,
    speed_std=speed_std_base,
)

print("Saved short-horizon strict dataset to:", SHORT_DATASET)

Base strict artifact loaded.
Base X: (2208, 1821, 6) Base Y: (2208, 1821)
Base IN_LEN: 24
Short-horizon strict window counts:
train: 1057
val:   337
test:  721
Saved short-horizon strict dataset to: artifacts/pems_graph_dataset_strict_short_h1_6_12_24.npz
